# Введение в MapReduce модель на Python


In [45]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [46]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [47]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [48]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [49]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [50]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [51]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [52]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [53]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [54]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [55]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [56]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [57]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [58]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str
    
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)
 
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [59]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])
 
def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])
      
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 2.9681156584664445),
 (1, 2.9681156584664445),
 (2, 2.9681156584664445),
 (3, 2.9681156584664445),
 (4, 2.9681156584664445)]

## Inverted index 

In [60]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)
      
def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)
 
def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [61]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [62]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()
      
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [63]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)
      
  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)
  
# try to set COMBINER=REDUCER and look at the number of values sent over the network 
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('is', 18), ('it', 18)]),
 (1, [('a', 2), ('what', 10)])]

## TeraSort

In [64]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for value in split:
        yield (value, None)
      
  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])
    
def MAP(value:int, _):
  yield (value, None)
  
def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, 0.007349357707748472),
   (None, 0.0956943725880397),
   (None, 0.13139763177873842),
   (None, 0.13159512007080942),
   (None, 0.1868266969338429),
   (None, 0.20585518025125837),
   (None, 0.3485602386341864),
   (None, 0.3608285254050878),
   (None, 0.38680196971387004),
   (None, 0.4232055093453443),
   (None, 0.4276560485007921),
   (None, 0.4294835519070682),
   (None, 0.4299350294848152),
   (None, 0.47215482219065585),
   (None, 0.481686250996057)]),
 (1,
  [(None, 0.5428391194866639),
   (None, 0.5544754743313279),
   (None, 0.6109653268067876),
   (None, 0.6203652520448144),
   (None, 0.6361384564754488),
   (None, 0.6495090576951329),
   (None, 0.7695105021207661),
   (None, 0.7824772793741013),
   (None, 0.8518234085815065),
   (None, 0.8980436362697071),
   (None, 0.9128236419170853),
   (None, 0.921037666383472),
   (None, 0.9277156392405052),
   (None, 0.9632858589709101),
   (None, 0.9840858083897736)])]

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [65]:
import random
from typing import Iterator, Tuple

numbers = [random.randint(0, 100) for _ in range(10)]
print("Сгенерированный список:", numbers)

def RECORDREADER() -> Iterator[Tuple[str, int]]:
    """
    Функция-ридер (RECORDREADER), итерируется по списку `numbers`
    и выдаёт кортеж (строковый индекс, число).
    """
    for idx, value in enumerate(numbers):
        yield f"{idx}", value

def MAP(num_id: str, num: int) -> Iterator[Tuple[int, int]]:
    """
    Функция MAP, которая принимает (ключ, число) и выдаёт (0, число)
    — то есть используем общий ключ 0 для всех чисел.
    """
    yield 0, num

def REDUCE(num_id: str, nums: Iterator[int]) -> Iterator[Tuple[str, int]]:
    """
    Функция REDUCE, которая принимает (ключ, итератор чисел),
    и возвращает (ключ, максимум из этих чисел).
    """
    yield num_id, max(nums)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)  # Преобразуем итератор в список
print("Результат:", output)


Сгенерированный список: [100, 3, 30, 92, 11, 50, 0, 38, 81, 50]
Результат: [(0, 100)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [66]:
numbers = [random.randint(0, 100) for _ in range(10)]
print("Сгенерированный список:", numbers)

def RECORDREADER() -> Iterator[Tuple[str, int]]:
    for idx, num in enumerate(numbers):
        yield f"{idx}", num

def MAP(num_id: str, num: int) -> Iterator[Tuple[int, int]]:
    yield 0, num

def REDUCE(num_id: str, nums: Iterator[int]) -> Iterator[Tuple[str, float]]:
    """
    Функция REDUCE: собирает итератор чисел в список,
    вычисляет среднее (сумма делённая на количество)
    и возвращает (ключ, среднее).
    """
    nums_list = list(nums)
    avg_value = sum(nums_list) / len(nums_list)
    yield num_id, avg_value

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
print("Результат (среднее значение):", output)


Сгенерированный список: [75, 42, 94, 26, 73, 52, 72, 52, 7, 24]
Результат (среднее значение): [(0, 51.7)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [67]:
from typing import List, Dict, Any

def groupbykey_sorted(iterable: List[Tuple[Any, Any]]) -> Iterator[Tuple[Any, List[Any]]]:
    """
    Группирует список кортежей (ключ, значение) по ключу, используя сортировку.
    Возвращает итератор по парам (ключ, список значений).

    :param iterable: исходный список кортежей (ключ, значение).
    :return: итератор по сгруппированным парам (ключ, список значений).
    """
    sorted_iterable = sorted(iterable, key=lambda x: x[0])
    grouped: Dict[Any, List[Any]] = {}
    current_key = None
    current_values: List[Any] = []

    for key, value in sorted_iterable:
        if key != current_key:
            # Если накопились значения для предыдущего ключа, сохраняем их
            if current_key is not None:
                grouped[current_key] = current_values
            current_key = key
            current_values = []
        current_values.append(value)

    # Добавляем последнюю группу после цикла
    if current_key is not None:
        grouped[current_key] = current_values

    return grouped.items()

example_1 = [('cat', 1), ('cat', 2), ('dog', 3), ('dog', 4), ('cat', 5)]
example_2 = [('apple', 'red'), ('banana', 'yellow'), ('apple', 'green'), ('banana', 'green'), ('cherry', 'red')]
example_3 = [('x', 10), ('y', 20), ('x', 30), ('z', 40), ('y', 50), ('z', 60)]

grouped_1 = groupbykey_sorted(example_1)
grouped_2 = groupbykey_sorted(example_2)
grouped_3 = groupbykey_sorted(example_3)

print("Пример 1 — исходные данные:", example_1)
print("Пример 1 — результат группировки:", list(grouped_1))

print("\nПример 2 — исходные данные:", example_2)
print("Пример 2 — результат группировки:", list(grouped_2))

print("\nПример 3 — исходные данные:", example_3)
print("Пример 3 — результат группировки:", list(grouped_3))


Пример 1 — исходные данные: [('cat', 1), ('cat', 2), ('dog', 3), ('dog', 4), ('cat', 5)]
Пример 1 — результат группировки: [('cat', [1, 2, 5]), ('dog', [3, 4])]

Пример 2 — исходные данные: [('apple', 'red'), ('banana', 'yellow'), ('apple', 'green'), ('banana', 'green'), ('cherry', 'red')]
Пример 2 — результат группировки: [('apple', ['red', 'green']), ('banana', ['yellow', 'green']), ('cherry', ['red'])]

Пример 3 — исходные данные: [('x', 10), ('y', 20), ('x', 30), ('z', 40), ('y', 50), ('z', 60)]
Пример 3 — результат группировки: [('x', [10, 30]), ('y', [20, 50]), ('z', [40, 60])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [68]:
from typing import Iterator, Tuple, Any

# Исходные данные и параметры
input_values = [1, 1, 1, 1, 2, 3, 3, 2, 3]
maps = 3
reducers = 2

def RECORDREADER(split: list[int]) -> Iterator[Tuple[int, None]]:
    for value in split:
        yield (value, None)

def INPUTFORMAT() -> Iterator[Iterator[Tuple[int, None]]]:
    """
    Функция, которая разбивает общий список данных (input_values)
    на подсписки (количество равно maps). Возвращает итератор,
    выдающий итераторы, созданные RECORDREADER.

    :return: Итератор итераторов, каждый из которых даёт (значение, None).
    """
    global maps
    split_size = int(np.ceil(len(input_values) / maps))
    for i in range(0, len(input_values), split_size):
        yield RECORDREADER(input_values[i : i + split_size])

def MAP(value: int, _: Any) -> Iterator[Tuple[int, None]]:
    yield (value, None)

def PARTITIONER(key: int, reducers: int = 2) -> int:
    """
    Функция PARTITIONER: определяет, в какой партиции (reducers) окажется ключ,
    используя хеш по модулю количества редьюсеров.

    :param key: Ключ (int).
    :param reducers: Число редьюсеров, по умолчанию 2.
    :return: Номер партиции (от 0 до reducers-1).
    """
    return hash(key) % reducers

def COMBINER(key: int, values: Iterator[Any]) -> Iterator[Tuple[int, None]]:
    """
    Функция COMBINER может осуществлять локальное сжатие данных перед REDUCE.
    В нашем случае просто возвращает (ключ, None).

    :param key: Ключ (int).
    :param values: Итератор значений.
    :return: Итератор (ключ, None).
    """
    yield (key, None)

def REDUCE(key: int, values: Iterator[Any]) -> Iterator[Tuple[int, None]]:
    """
    Функция REDUCE: сводит значения для данного ключа.
    Здесь мы ничего не считаем, просто возвращаем (ключ, None).

    :param key: Ключ (int).
    :param values: Итератор значений.
    :return: Итератор (ключ, None).
    """
    yield (key, None)

partitioned_output = MapReduceDistributed(
    INPUTFORMAT,
    MAP,
    REDUCE,
    COMBINER=None,          # COMBINER можно включить при необходимости
    PARTITIONER=PARTITIONER
)

# Собираем результат в список, чтобы показать итоговую структуру по партициям
partitioned_output = [
    (partition_id, list(partition_data))
    for (partition_id, partition_data) in partitioned_output
]

# Выводим результат
print("Результат партиционирования:", partitioned_output)


9 key-value pairs were sent over a network.
Результат партиционирования: [(0, [(2, None)]), (1, [(1, None), (3, None)])]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [69]:
import random
from typing import Iterator, Tuple, Any

# Генерируем список случайных чисел
numbers = [random.randint(0, 1000) for _ in range(10)]
print("Сгенерированные числа:", numbers)

# Лямбда-функция, определяющая условие (проверяем чётность)
pred = lambda x: x % 2 == 0

def RECORDREADER() -> Iterator[Tuple[str, int]]:
    for idx, value in enumerate(numbers):
        yield f"{idx}", value

def MAP(num_id: str, value: int) -> Iterator[Tuple[int, int]]:
    """
    Функция MAP:
    - Принимает (строковый индекс, значение);
    - Если `value` удовлетворяет предикату (чётное число), возвращает (value, value).
    - Иначе не возвращает ничего (пропускает нечетные числа).
    """
    if pred(value):
        yield value, value

def REDUCE(key: int, values: Iterator[int]) -> Iterator[Tuple[int, list[int]]]:
    vals_list = list(values)
    yield key, vals_list

output = MapReduce(RECORDREADER, MAP, REDUCE)

output_list = list(output)
print("Результат MapReduce:", output_list)


Сгенерированные числа: [250, 197, 738, 764, 59, 560, 884, 416, 548, 136]
Результат MapReduce: [(250, [250]), (738, [738]), (764, [764]), (560, [560]), (884, [884]), (416, [416]), (548, [548]), (136, [136])]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [70]:
import random
from typing import NamedTuple, List, Tuple, Iterator

class User(NamedTuple):
    id: int
    age: int
    social_contacts: int
    gender: str

input_collection = [
    User(id=0, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000)),
    User(id=1, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000)),
    User(id=2, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000)),
    User(id=3, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000))
]

print("Сгенерированные пользователи:")
for user in input_collection:
    print(user)

# Атрибуты для проекции
attributes = ['age', 'gender']

def MAP(_, row: NamedTuple) -> Iterator[Tuple[Tuple, Tuple]]:
    """
    Функция MAP:
    - Принимает строку данных и делает проекцию по атрибутам, указанным в `attributes`.
    """
    projection = tuple(getattr(row, attribute) for attribute in attributes)
    yield (projection, projection)

def REDUCE(proj: NamedTuple, _) -> Iterator[Tuple[NamedTuple, NamedTuple]]:
    yield (proj, proj)

def RECORDREADER() -> List[Tuple[int, NamedTuple]]:
    """
    Функция RECORDREADER:
    - Возвращает кортежи (id, user), где user — это объект класса User.
    """
    return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)

output_list = list(output)
print("\nРезультат MapReduce:", output_list)


Сгенерированные пользователи:
User(id=0, age=38, social_contacts=226, gender='female')
User(id=1, age=49, social_contacts=521, gender='female')
User(id=2, age=41, social_contacts=775, gender='female')
User(id=3, age=47, social_contacts=611, gender='female')

Результат MapReduce: [((38, 'female'), (38, 'female')), ((49, 'female'), (49, 'female')), ((41, 'female'), (41, 'female')), ((47, 'female'), (47, 'female'))]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [71]:
input_collection1 = [
    User(id=0, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000)),
    User(id=1, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000)),
    User(id=2, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000)),
    User(id=3, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000))
]

input_collection2 = [
    User(id=0, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000)),
    User(id=1, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000)),
    User(id=2, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000)),
    User(id=3, age=random.randint(18, 70), gender=random.choice(['male', 'female']), social_contacts=random.randint(5, 1000))
]

print("Сгенерированные пользователи:")
for user in input_collection1:
    print(user)
for user in input_collection2:
    print(user)

def MAP(_, value: User) -> Iterator[Tuple[User, User]]:
    yield (value, value)

def REDUCE(id: int, values: Iterator[User]) -> Iterator[Tuple[int, int]]:
    yield (id, id)

def RECORDREADER() -> List[Tuple[int, User]]:
    """
    Функция RECORDREADER: объединяет два списка пользователей в один, возвращая
    кортежи вида (id, User), где id — это идентификатор пользователя.

    :return: Список кортежей (id, User).
    """
    return [(u.id, u) for u in input_collection1] + [(u.id, u) for u in input_collection2]

output = MapReduce(RECORDREADER, MAP, REDUCE)

output_list = list(output)
print("\nРезультат MapReduce:\n")
output_list

Сгенерированные пользователи:
User(id=0, age=63, social_contacts=462, gender='female')
User(id=1, age=48, social_contacts=749, gender='female')
User(id=2, age=59, social_contacts=860, gender='female')
User(id=3, age=21, social_contacts=252, gender='male')
User(id=0, age=35, social_contacts=359, gender='male')
User(id=1, age=29, social_contacts=450, gender='female')
User(id=2, age=31, social_contacts=278, gender='male')
User(id=3, age=42, social_contacts=261, gender='female')

Результат MapReduce:



[(User(id=0, age=63, social_contacts=462, gender='female'),
  User(id=0, age=63, social_contacts=462, gender='female')),
 (User(id=1, age=48, social_contacts=749, gender='female'),
  User(id=1, age=48, social_contacts=749, gender='female')),
 (User(id=2, age=59, social_contacts=860, gender='female'),
  User(id=2, age=59, social_contacts=860, gender='female')),
 (User(id=3, age=21, social_contacts=252, gender='male'),
  User(id=3, age=21, social_contacts=252, gender='male')),
 (User(id=0, age=35, social_contacts=359, gender='male'),
  User(id=0, age=35, social_contacts=359, gender='male')),
 (User(id=1, age=29, social_contacts=450, gender='female'),
  User(id=1, age=29, social_contacts=450, gender='female')),
 (User(id=2, age=31, social_contacts=278, gender='male'),
  User(id=2, age=31, social_contacts=278, gender='male')),
 (User(id=3, age=42, social_contacts=261, gender='female'),
  User(id=3, age=42, social_contacts=261, gender='female'))]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [72]:

def MAP(_, value: User) -> Iterator[Tuple[User, User]]:
    yield (value, value)

def REDUCE(id: int, values: Iterator[User]) -> Iterator[Tuple[int, int]]:
    """
    Функция REDUCE:
    - Если для данного идентификатора найдено два значения, то выводится кортеж (id, id).

    :param id: Идентификатор пользователя.
    :param values: Итератор значений, связанных с этим id.
    :return: кортеж (id, id), если количество значений равно 2.
    """
    values_list = list(values)
    if len(values_list) == 2:
        yield (id, id)

def RECORDREADER() -> List[Tuple[int, User]]:
    """
    Функция RECORDREADER:
    - Объединяет два списка пользователей и возвращает их как кортежи (id, user).

    :return: Список кортежей (id, user).
    """
    return [(u.id, u) for u in input_collection1] + [(u.id, u) for u in input_collection2]

output = MapReduce(RECORDREADER, MAP, REDUCE)
print("\nРезультат MapReduce:\n")
output_list



Результат MapReduce:



[(User(id=0, age=63, social_contacts=462, gender='female'),
  User(id=0, age=63, social_contacts=462, gender='female')),
 (User(id=1, age=48, social_contacts=749, gender='female'),
  User(id=1, age=48, social_contacts=749, gender='female')),
 (User(id=2, age=59, social_contacts=860, gender='female'),
  User(id=2, age=59, social_contacts=860, gender='female')),
 (User(id=3, age=21, social_contacts=252, gender='male'),
  User(id=3, age=21, social_contacts=252, gender='male')),
 (User(id=0, age=35, social_contacts=359, gender='male'),
  User(id=0, age=35, social_contacts=359, gender='male')),
 (User(id=1, age=29, social_contacts=450, gender='female'),
  User(id=1, age=29, social_contacts=450, gender='female')),
 (User(id=2, age=31, social_contacts=278, gender='male'),
  User(id=2, age=31, social_contacts=278, gender='male')),
 (User(id=3, age=42, social_contacts=261, gender='female'),
  User(id=3, age=42, social_contacts=261, gender='female'))]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [73]:

def MAP(_, value: User, collection: str) -> Iterator[Tuple[User, str]]:
    """
    Функция MAP:
    - Принимает объект User и строку, определяющую коллекцию.
    - Если коллекция '1', метка 'R', если '2' — метка 'S'.

    :param _: Игнорируем первый параметр.
    :param value: Объект класса User.
    :param collection: Строка, указывающая, к какой коллекции принадлежит объект.
    :return: Итератор с кортежами (value, 'R' или 'S').
    """
    if collection == '1':
        yield (value, 'R')
    else:
        yield (value, 'S')

def REDUCE(id: int, values: Iterator[str]) -> Iterator[Tuple[int, int]]:
    """
    Функция REDUCE:
    - Для каждого id проверяет, все ли значения равны 'R'.
    - Если это так, возвращает (id, id).

    :param id: Идентификатор пользователя.
    :param values: Итератор строковых значений ('R' или 'S').
    :return: Кортеж (id, id), если все значения 'R'.
    """
    if all(i == 'R' for i in values):
        yield (id, id)

def RECORDREADER() -> List[Tuple[int, User, str]]:
    """
    Функция RECORDREADER:
    - Создаёт список кортежей, состоящих из (id, user, коллекция), где '1' или '2'.
    :return: Список кортежей (id, user, коллекция).
    """
    return [(u.id, u, '1') for u in input_collection1] + [(u.id, u, '2') for u in input_collection2]

output = MapReduce(RECORDREADER, MAP, REDUCE)

output_list = list(output)
print("\nРезультат MapReduce:\n")
output_list


Результат MapReduce:



[(User(id=0, age=63, social_contacts=462, gender='female'),
  User(id=0, age=63, social_contacts=462, gender='female')),
 (User(id=1, age=48, social_contacts=749, gender='female'),
  User(id=1, age=48, social_contacts=749, gender='female')),
 (User(id=2, age=59, social_contacts=860, gender='female'),
  User(id=2, age=59, social_contacts=860, gender='female')),
 (User(id=3, age=21, social_contacts=252, gender='male'),
  User(id=3, age=21, social_contacts=252, gender='male'))]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [80]:
from typing import NamedTuple, List, Tuple, Iterator


class Employee(NamedTuple):
    """
    Класс сотрудника, где:
    - id: уникальный идентификатор,
    - salary: зарплата (целое число).
    """
    id: int
    salary: int

num_users = 4
num_employees = 4

input_collection1 = [
    User(
        id=i,
        age=random.randint(18, 70),
        gender=random.choice(['male', 'female']),
        social_contacts=random.randint(0, 2000)
    )
    for i in range(num_users)
]

input_collection2 = [
    Employee(
        id=i,
        salary=random.randint(200, 10_000)
    )
    for i in range(num_employees)
]

print("Сгенерированные пользователи (input_collection1):")
for user in input_collection1:
    print(user)

print("\nСгенерированные сотрудники (input_collection2):")
for emp in input_collection2:
    print(emp)

def MAP(_: int, value: NamedTuple, collection: str) -> Iterator[Tuple[int, Tuple[str, str]]]:
    """
    Функция MAP:
      - Принимает кортеж (индекс, значение, коллекция), но индекс игнорируется (поэтому _).
      - Если коллекция == '1', генерирует (value.id, ('R', value.gender)).
      - Если коллекция == '2', генерирует (value.id, ('S', value.salary)).

    :param _: Игнорируемый индекс.
    :param value: Объект класса User или Employee (NamedTuple).
    :param collection: '1' или '2', указывающая, к какой коллекции относится запись.
    :return: Итератор кортежей (ключ, (метка, значение)).
    """
    if collection == '1':
        yield (value.id, ('R', value.gender))
    else:
        yield (value.id, ('S', str(value.salary)))  # переводим в str для единообразия

def REDUCE(id_value: int, values: Iterator[Tuple[str, str]]) -> Iterator[Tuple[str, int, str]]:
    """
    Функция REDUCE:
      - Группирует значения по идентификатору (id_value).
      - Разделяет на списки S и R, исходя из первой части кортежа ('S' или 'R').
      - Для каждого значения из S_list и R_list генерирует кортеж (salary, id_value, gender).

    :param id_value: Идентификатор (int), общий для User и Employee.
    :param values: Итератор, содержащий пары ('R'/'S', значение).
    :return: Итератор (salary, id_value, gender) для каждого сочетания S и R.
    """
    s_list = [val for (tag, val) in values if tag == 'S']
    r_list = [val for (tag, val) in values if tag == 'R']

    for salary in s_list:
        for gender in r_list:
            # Возвращаем кортеж (salary, id, gender)
            yield (salary, id_value, gender)

def RECORDREADER() -> List[Tuple[int, NamedTuple, str]]:
    """
    Функция RECORDREADER:
      - Создаёт объединённый список кортежей (id, объект, '1' или '2'),
        где '1' означает, что запись из input_collection1 (User),
        а '2' — что запись из input_collection2 (Employee).

    :return: Список кортежей (id, user/employee, коллекция).
    """
    return [
        (u.id, u, '1') for u in input_collection1
    ] + [
        (e.id, e, '2') for e in input_collection2
    ]

output = MapReduce(RECORDREADER, MAP, REDUCE)

output_list = list(output)
print("\nРезультат MapReduce:\n")
output_list


Сгенерированные пользователи (input_collection1):
User(id=0, age=60, social_contacts=1981, gender='female')
User(id=1, age=53, social_contacts=119, gender='female')
User(id=2, age=53, social_contacts=632, gender='male')
User(id=3, age=26, social_contacts=1053, gender='male')

Сгенерированные сотрудники (input_collection2):
Employee(id=0, salary=746)
Employee(id=1, salary=3511)
Employee(id=2, salary=7964)
Employee(id=3, salary=6574)

Результат MapReduce:



[('746', 0, 'female'),
 ('3511', 1, 'female'),
 ('7964', 2, 'male'),
 ('6574', 3, 'male')]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [81]:
import random
from typing import NamedTuple, List, Tuple, Iterator


NUM_USERS = 5
input_collection = [
    User(
        id=i,
        age=random.randint(18, 70),
        gender=random.choice(["male", "female"]),
        social_contacts=random.randint(0, 2000)
    )
    for i in range(NUM_USERS)
]

# Функция, которую будем применять в REDUCE для суммирования
op = sum

def MAP(_, row: User) -> Iterator[Tuple[str, int]]:
    """
    Функция MAP:
    - Для каждого пользователя отдаёт пару (gender, social_contacts).
    """
    yield (row.gender, row.social_contacts)

def REDUCE(gender: str, values: Iterator[int]) -> Iterator[Tuple[str, int]]:
    """
    Функция REDUCE:
    - Принимает ключ (gender) и итератор значений (social_contacts).
    - Суммирует все контакты по ключу (gender) и возвращает результат.
    """
    yield (gender, op(values))

def RECORDREADER() -> List[Tuple[int, User]]:
    """
    Функция RECORDREADER:
    - Превращает список `input_collection` в список пар (id, User).
    - Возвращает список (id, user).
    """
    return [(u.id, u) for u in input_collection]

print("Случайно сгенерированные пользователи:")
for user in input_collection:
    print(user)


output = MapReduce(RECORDREADER, MAP, REDUCE)
output_list = list(output)

print("\nРезультат MapReduce (сумма social_contacts по полу):")
for item in output_list:
    print(item)


Случайно сгенерированные пользователи:
User(id=0, age=39, social_contacts=1473, gender='male')
User(id=1, age=68, social_contacts=797, gender='female')
User(id=2, age=55, social_contacts=834, gender='male')
User(id=3, age=60, social_contacts=1866, gender='male')
User(id=4, age=22, social_contacts=1645, gender='male')

Результат MapReduce (сумма social_contacts по полу):
('male', 5818)
('female', 797)


# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [82]:
from typing import Iterator, Tuple
import numpy as np
import math

# Генерация случайных данных
ROWS, COLS = 5, 4
mat = np.random.rand(ROWS, COLS)  # Матрица размера (ROWS x COLS)
vec = np.random.rand(COLS)        # Вектор размера (COLS)

def MAP(coordinates: Tuple[int, int], value: float) -> Iterator[Tuple[int, Tuple[int, float]]]:
    """
    Функция MAP:
      - Принимает координаты (i, j) и значение (float).
      - Возвращает кортеж (i, (j, value)), чтобы затем по i сгруппировать все элементы одной строки.
    """
    i, j = coordinates
    yield i, (j, value)

def REDUCE(i: int, products: Iterator[Tuple[int, float]]) -> Iterator[Tuple[int, float]]:
    """
    Функция REDUCE:
      - Получает ключ i (номер строки) и итерируемые пары (j, value).
      - Суммирует произведения для всех j в этой строке,
        используя math.prod (вместо умножения между элементами матрицы и элемента вектора).
      - Возвращает (i, сумма) как результат для i-й строки.
    """
    # Преобразуем итератор в список, чтобы могли проходить несколько раз
    products_list = list(products)
    total = 0
    for j in range(mat.shape[1]):
        # Извлекаем все значения, соответствующие тому же j
        # и берём их произведение (так как одна запись приходит из mat, другая — из vec)
        total += math.prod(v for (jj, v) in products_list if jj == j)
    yield i, total

def RECORDREADER() -> Iterator[Tuple[Tuple[int, int], float]]:
    """
    Функция RECORDREADER:
      - Перебирает все элементы матрицы 'mat' и вектора 'vec',
      - Для каждой пары (i, j) сначала выдаёт ( (i, j), mat[i,j] ),
        а затем ( (i, j), vec[j] ).
    """
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            yield ( (i, j), mat[i, j] )
            yield ( (i, j), vec[j] )

output = MapReduce(RECORDREADER, MAP, REDUCE)
output_list = list(output)

print("Случайная матрица (mat):")
print(mat)
print("\nСлучайный вектор (vec):")
print(vec)

print("\nРезультат MapReduce (матрица * вектор):")
for item in output_list:
    print(item)


Случайная матрица (mat):
[[0.68429638 0.75053995 0.84538706 0.68255275]
 [0.46834066 0.64099964 0.90373152 0.73495398]
 [0.99190906 0.19716962 0.03707393 0.93738958]
 [0.23009795 0.44224335 0.32578055 0.37905474]
 [0.37741502 0.80720982 0.90566768 0.45162852]]

Случайный вектор (vec):
[0.21240356 0.12170762 0.92884222 0.14841234]

Результат MapReduce (матрица * вектор):
(0, 1.123223860996149)
(1, 1.1259919915416143)
(2, 0.40823806777199834)
(3, 0.4615531421336464)
(4, 1.0866575072725255)


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [84]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [83]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])
      
def MAP(k1, v1):
	(j, k) = k1
	w = v1
	for i in range(I):
		yield ((i, k), (w * small_mat[i,j]))

def REDUCE(key, values):
  (i, k) = key
  yield (key, sum(values))

Проверьте своё решение

In [85]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat) 
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [86]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [87]:
import numpy as np
import math
from typing import Iterator, Tuple, Any

# Размеры матриц (I x J) и (J x K)
I, J, K = 2, 3, 4 * 10

# Генерируем случайные матрицы M (I x J) и N (J x K)
M = np.random.rand(I, J)
N = np.random.rand(J, K)

def RECORDREADER() -> Iterator[Tuple[Tuple[int, int], float, str]]:
    """
    Функция RECORDREADER:
      - Итерируется по всем индексам j (от 0 до J-1).
      - Для каждого j и k (от 0 до K-1) отдаёт ((j, k), N[j, k], 'N').
      - Для каждого j и i (от 0 до I-1) отдаёт ((i, j), M[i, j], 'M').

    Таким образом, создаются "записи", которые могут быть распределённо обработаны:
      - ( (j, k), значение_из_N, 'N' )
      - ( (i, j), значение_из_M, 'M' )
    """
    for j in range(J):
        # Сначала элементы строки j матрицы N
        for k in range(K):
            yield ((j, k), N[j, k], 'N')
        # Затем элементы столбца j матрицы M
        for i in range(I):
            yield ((i, j), M[i, j], 'M')

def MAP(k1: Tuple[int, int], v1: float, t: str) -> Iterator[Tuple[Tuple[int, int], Tuple[int, float]]]:
    """
    Функция MAP:
      - Если запись помечена 'N', значит это элемент N[j, k].
        Тогда для всех i (0..I-1) формируем ключ (i, k) и значение (j, w).
      - Если запись помечена 'M', значит это элемент M[i, j].
        Тогда для всех k (0..K-1) формируем ключ (i, k) и значение (j, v).

    :param k1: Ключ (индексы), (j, k) или (i, j).
    :param v1: Значение (число из M или N).
    :param t: Метка 'N' или 'M', указывающая, из какой матрицы взято число.
    :return: Итератор ( (i, k), (j, значение) ).
    """
    if t == 'N':
        (j, k) = k1
        w = v1
        for i in range(I):
            yield ( (i, k), (j, w) )
    else:  # t == 'M'
        (i, j) = k1
        v = v1
        for k in range(K):
            yield ( (i, k), (j, v) )

def REDUCE(key: Tuple[int, int], values: Iterator[Tuple[int, float]]) -> Iterator[Tuple[Tuple[int, int], float]]:
    """
    Функция REDUCE:
      - Принимает ключ (i, k) и итерируемые пары (j, значение).
      - Суммирует произведения тех значений, у которых j совпадает (т.е. одно из M и одно из N).
      - Возвращает ( (i, k), result ), где result = (i-я строка M) * (k-й столбец N).

    :param key: Кортеж (i, k).
    :param values: Итератор пар (j, значение), сгруппированных по (i, k).
    :return: Итератор ( (i, k), результат_умножения ).
    """
    (i, k) = key
    # Превращаем итератор в список, чтобы можно было несколько раз по нему пройти
    all_values = list(values)
    result = 0
    for j in range(J):
        # Извлекаем все элементы, у которых индекс j совпадает
        # Предполагается, что придёт одно значение из M и одно из N для каждого j
        # их произведение суммируем
        pair_values = [val for (jj, val) in all_values if jj == j]
        if len(pair_values) == 2:
            # Предположим, это два числа, одно из M, одно из N
            # Можем взять их произведение
            result += math.prod(pair_values)
        # Если пришло меньше или больше, зависит от структуры данных
        # (В корректном случае должно быть ровно 2)
    yield (key, result)

# Для проверки вычислим эталонное решение
reference_solution = np.matmul(M, N)

solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output: Iterator[Tuple[Tuple[int, int], float]]) -> np.ndarray:
    """
    Вспомогательная функция:
      - Превращает выход REDUCE, который представляет собой последовательность
        ( (i, k), элемент_результата ), в матрицу numpy размером (I x K).

    :param reduce_output: Итератор кортежей ( (i, k), value ).
    :return: numpy-массив размером (I x K).
    """
    reduce_list = list(reduce_output)
    I_max = max(i for ((i, _), _) in reduce_list) + 1
    K_max = max(k for ((_, k), _) in reduce_list) + 1
    mat = np.empty(shape=(I_max, K_max))
    for ((i, k), val) in reduce_list:
        mat[i, k] = val
    return mat

result_matrix = asmatrix(solution)
print("Сгенерированная матрица M:\n", M)
print("\nСгенерированная матрица N:\n", N)

print("\nРезультат умножения (MapReduce):\n", result_matrix)
print("\nЭталонное решение (numpy.matmul):\n", reference_solution)

# Проверим, совпадают ли результаты
print("\nСовпадает ли решение:", np.allclose(reference_solution, result_matrix))


Сгенерированная матрица M:
 [[0.6757294  0.23613324 0.83698429]
 [0.75026745 0.95173292 0.54765005]]

Сгенерированная матрица N:
 [[0.66145231 0.19313818 0.9542943  0.39335599 0.88792481 0.49555303
  0.52634234 0.53389347 0.10625755 0.49323671 0.8976906  0.61462634
  0.64341127 0.05665185 0.85295182 0.45389703 0.88382779 0.37276727
  0.11959751 0.12994728 0.00951608 0.00385157 0.41647474 0.37008011
  0.71265473 0.07524081 0.23537496 0.35398202 0.13175415 0.931579
  0.38485243 0.51756273 0.27887888 0.260304   0.1899843  0.74510203
  0.39081938 0.90452689 0.49387473 0.017086  ]
 [0.61839171 0.76593926 0.99577679 0.46982851 0.43829435 0.64036055
  0.94195834 0.55360134 0.53977052 0.92953865 0.46725322 0.88543605
  0.54469008 0.95165717 0.89143538 0.91935458 0.96208354 0.91358945
  0.37460212 0.16109318 0.38776353 0.72066818 0.09824476 0.24630946
  0.08801316 0.04809674 0.96230362 0.48134778 0.55785017 0.1672656
  0.07915007 0.87778075 0.48035022 0.54172626 0.87023033 0.28245514
  0.683995

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [88]:
import math
import random
import numpy as np
from typing import Iterator, Tuple, Any

I, J, K = 2, 3, 4 * 10

M = np.random.rand(I, J)
N = np.random.rand(J, K)

maps = 2
reducers = 2

def PARTITIONER(key: Tuple[int, int], reducers: int = 2) -> int:
    """
    Функция PARTITIONER:
      - Определяет, в какой раздел (partition_id) отправится ключ.
      - Здесь используется простое хеширование по модулю количества редьюсеров.
    """
    return hash(key) % reducers

def RECORDREADER(ind1: int, ind2: int, mark: str) -> Iterator[Tuple[Tuple[int, int], float, str]]:
    """
    Генератор (ридер) данных:
      - Для заданных размеров ind1, ind2 и метки mark ('M' или 'N')
      - Если mark == 'M', выдаёт элементы M[i,j] с ключом (i,j).
      - Если mark == 'N', выдаёт элементы N[i,j] с ключом (i,j).

    :param ind1: Первая размерность (зависит от M или N).
    :param ind2: Вторая размерность (зависит от M или N).
    :param mark: 'M' или 'N' (указывается, из какой матрицы берутся данные).
    :return: итератор кортежей ((i,j), значение, 'M'/'N').
    """
    if mark == 'M':
        # Матрица M имеет размер I x J
        # Перебираем i, j
        for i in range(ind1):
            for j in range(ind2):
                yield ((i, j), float(M[i, j]), mark)
    else:  # mark == 'N'
        # Матрица N имеет размер J x K
        # Перебираем j, k
        for j in range(ind1):
            for k in range(ind2):
                yield ((j, k), float(N[j, k]), mark)

def INPUTFORMAT() -> Iterator[Iterator[Tuple[Tuple[int, int], float, str]]]:
    """
    Функция INPUTFORMAT:
      - Определяет список "ридеров" (итераторов) данных (шарды),
        которые будут раздаваться мапперам.
      - Возвращает итераторы:
        1) RECORDREADER для M (размеры I, J, пометка 'M')
        2) RECORDREADER для N (размеры J, K, пометка 'N')

    :return: итератор итераторов, каждый из которых возвращает кортеж
             ((i,j), значение, 'M'/'N').
    """
    # В данном примере maps=2 не даёт реального разбиения, но иллюстрирует идею.
    yield RECORDREADER(I, J, 'M')
    yield RECORDREADER(J, K, 'N')

def MAP(
    k1: Tuple[int, int],
    v1: float,
    t: str
) -> Iterator[Tuple[Tuple[int, int], Tuple[int, float]]]:
    """
    Функция MAP:
      - Если t == 'N', имеем элемент N[j, k] => для всех i формируем ключ (i, k)
        и значение (j, v1).
      - Если t == 'M', имеем элемент M[i, j] => для всех k формируем ключ (i, k)
        и значение (j, v1).

    :param k1: Ключ (i, j) или (j, k) в зависимости от матрицы.
    :param v1: Значение элемента из матрицы M или N.
    :param t: Метка 'M' или 'N'.
    :return: Итератор ( (i, k), (j, v) ).
    """
    if t == 'N':
        (j, k) = k1
        w = v1
        for i in range(I):
            yield ((i, k), (j, w))
    else:  # t == 'M'
        (i, j) = k1
        v = v1
        for k in range(K):
            yield ((i, k), (j, v))

def REDUCE(
    key: Tuple[int, int],
    values: Iterator[Tuple[int, float]]
) -> Iterator[Tuple[Tuple[int, int], float]]:
    """
    Функция REDUCE:
      - Принимает ключ (i, k) и итератор пар (j, val).
      - Суммирует произведения для M[i,j] и N[j,k] (где j совпадает).
      - Возвращает ( (i,k), result ), где result — элемент произведения матриц.

    :param key: Кортеж (i, k).
    :param values: Итератор пар (j, значение).
    :return: ( (i, k), результат ) — элемент итоговой матрицы.
    """
    i, k = key
    all_vals = list(values)
    result = 0
    for j in range(J):
        # Найдём все пары, где jj == j
        pair_vals = [val for (jj, val) in all_vals if jj == j]
        # Предполагается, что pair_vals содержит 2 числа: из M и из N.
        # Если реально приходит больше/меньше, это ошибка входных данных.
        if len(pair_vals) == 2:
            result += pair_vals[0] * pair_vals[1]
    yield (key, result)

# Эталонное решение с помощью NumPy
reference_solution = np.matmul(M, N)

partitioned_output = MapReduceDistributed(
    INPUTFORMAT,
    MAP,
    REDUCE,
    COMBINER=None,
    PARTITIONER=PARTITIONER
)

partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

def asmatrix(reduce_output: list[Tuple[int, list[Tuple[Tuple[int, int], float]]]]) -> np.ndarray:
    """
    Преобразует выход распределённого редьюса в numpy-массив (I x K).

    :param reduce_output: список вида [(partition_id, [((i,k), value), ...]), ...].
    :return: Матрица numpy размером (I x K).
    """
    mat = np.empty(shape=(I, K))
    # Проходим по всем партициям
    for (_, partition_data) in reduce_output:
        for ((i, k), vw) in partition_data:
            mat[i, k] = vw
    return mat

result_matrix = asmatrix(partitioned_output)

print("Матрица M:\n", M)
print("\nМатрица N:\n", N)
print("\nРезультат распределённого MapReduce (M * N):\n", result_matrix)
print("\nЭталонное умножение NumPy:\n", reference_solution)

# Проверка на близость результатов
print("\nРезультаты совпадают?", np.allclose(reference_solution, result_matrix))


480 key-value pairs were sent over a network.
Матрица M:
 [[0.80530477 0.89922363 0.76907935]
 [0.02327184 0.64622212 0.8260522 ]]

Матрица N:
 [[0.75042481 0.63703659 0.47449515 0.38443548 0.94952884 0.29676037
  0.74669282 0.20942215 0.4048388  0.57694032 0.59383882 0.31668017
  0.97223671 0.98855958 0.55889734 0.00331766 0.61298548 0.76761527
  0.83335844 0.60006205 0.41490072 0.86757393 0.16747056 0.4156604
  0.23940269 0.13089294 0.55036418 0.52171086 0.50363764 0.93993117
  0.41660055 0.00128746 0.47779032 0.40221014 0.9925108  0.75020563
  0.30400382 0.91049891 0.84069407 0.09778608]
 [0.45544887 0.7657958  0.15607461 0.5734532  0.11053449 0.60298153
  0.48012391 0.0153709  0.66833799 0.32704499 0.81288886 0.68910337
  0.42927762 0.72706885 0.4615247  0.51217737 0.44745236 0.47767351
  0.27529471 0.33342339 0.18624764 0.90657333 0.17567494 0.14448844
  0.84539081 0.10502222 0.12270094 0.08330546 0.5021791  0.16247331
  0.30647913 0.96798959 0.51710729 0.36658702 0.49252029 0.809

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [89]:
import math
import random
import numpy as np
from typing import Iterator, Tuple, List

I, J, K = 2, 3, 4 * 10

M = np.random.rand(I, J)
N = np.random.rand(J, K)

# Параметры для распределённой обработки
maps = 5       # количество "шардов" (частей) на стадии map
reducers = 3   # количество редьюсеров

def PARTITIONER(key: Tuple[int, int], reducers: int = 3) -> int:
    """
    Функция PARTITIONER:
      - Определяет, в какой раздел (partition) отправляется ключ (i, k).
      - Здесь используется простое хэширование по модулю количества редьюсеров.
    """
    return hash(key) % reducers

def RECORDREADER(indices: List[Tuple[int, int]], mark: str) -> Iterator[Tuple[Tuple[int, int], float, str]]:
    """
    Функция RECORDREADER для чтения части (шарда) данных:
      - Принимает список индексов (i, j) для M или (j, k) для N, а также метку 'M'/'N'.
      - Возвращает кортеж ( (i, j)/(j, k), значение, mark ) для каждого индекса.

    :param indices: список кортежей (i, j) или (j, k).
    :param mark: 'M' или 'N', указывающая, из какой матрицы берутся данные.
    :return: Итератор кортежей (ключ, значение, метка).
    """
    for (x, y) in indices:
        if mark == 'M':
            # x -> i, y -> j
            yield ((x, y), float(M[x, y]), mark)
        else:
            # x -> j, y -> k
            yield ((x, y), float(N[x, y]), mark)

def INPUTFORMAT() -> Iterator[Iterator[Tuple[Tuple[int, int], float, str]]]:
    """
    Функция INPUTFORMAT:
      - Разбивает индексы для матрицы M (i, j) на M_maps частей,
        и индексы для матрицы N (j, k) на N_maps частей.
      - Возвращает соответствующие итераторы RECORDREADER.

    :return: Итератор итераторов, каждый из которых даёт ( (i,j)/(j,k), значение, 'M'/'N' ).
    """
    global maps
    # Определяем, сколько "шардов" отдать для M и сколько для N
    M_maps = maps // 2  # Часть шардов, используемых под M
    N_maps = maps - M_maps  # Остальные под N

    # Размер "шарда" для M
    M_split_size = (I * J) // M_maps if M_maps > 0 else I * J
    # Размер "шарда" для N
    N_split_size = (J * K) // N_maps if N_maps > 0 else J * K

    # Создаём список индексов для M: (i, j)
    M_indices = [(i, j) for i in range(I) for j in range(J)]
    np.random.shuffle(M_indices)  # Перемешиваем для наглядности

    # Создаём список индексов для N: (j, k)
    N_indices = [(j, k) for j in range(J) for k in range(K)]
    np.random.shuffle(N_indices)

    # Создаём итераторы для матрицы M
    for i in range(0, len(M_indices), M_split_size):
        yield RECORDREADER(M_indices[i : i + M_split_size], 'M')

    # Создаём итераторы для матрицы N
    for i in range(0, len(N_indices), N_split_size):
        yield RECORDREADER(N_indices[i : i + N_split_size], 'N')

def MAP(
    k1: Tuple[int, int],
    v1: float,
    t: str
) -> Iterator[Tuple[Tuple[int, int], Tuple[int, float]]]:
    """
    Функция MAP:
      - Если t == 'N' => ключ (j, k), значение v1 = N[j, k].
        Генерируем ((i, k), (j, v1)) для i in [0..I-1].
      - Если t == 'M' => ключ (i, j), значение v1 = M[i, j].
        Генерируем ((i, k), (j, v1)) для k in [0..K-1].

    :param k1: (i, j) или (j, k), зависящее от матрицы.
    :param v1: Значение элемента матрицы (float).
    :param t: Метка 'M' или 'N'.
    :return: Итератор ( (i, k), (j, value) ).
    """
    if t == 'N':
        (j, k) = k1
        for i in range(I):
            yield ((i, k), (j, v1))
    else:  # t == 'M'
        (i, j) = k1
        for k in range(K):
            yield ((i, k), (j, v1))

def REDUCE(
    key: Tuple[int, int],
    values: Iterator[Tuple[int, float]]
) -> Iterator[Tuple[Tuple[int, int], float]]:
    """
    Функция REDUCE:
      - Ключ (i, k).
      - Из values собираем пары (j, число) для всех j.
      - Находим сумму произведений M[i,j] * N[j,k].
        (В values придут данные и от M, и от N — нужно связать по одинаковым j).

    :param key: (i, k).
    :param values: Пары (j, value).
    :return: ( (i, k), сумма_произведений ).
    """
    i, k = key
    pairs_list = list(values)

    result = 0
    # Для каждого j суммируем произведения
    for j in range(J):
        # Выбираем все значения, у которых jj == j
        same_j_vals = [val for (jj, val) in pairs_list if jj == j]
        # Ожидаем, что там будут 2 числа: одно из M и одно из N
        if len(same_j_vals) == 2:
            result += same_j_vals[0] * same_j_vals[1]
    yield (key, result)

reference_solution = np.matmul(M, N)

partitioned_output = MapReduceDistributed(
    INPUTFORMAT,
    MAP,
    REDUCE,
    COMBINER=None,
    PARTITIONER=PARTITIONER
)

partitioned_output = [
    (partition_id, list(partition))
    for (partition_id, partition) in partitioned_output
]

def asmatrix(reduce_output: List[Tuple[int, List[Tuple[Tuple[int, int], float]]]]) -> np.ndarray:
    """
    Вспомогательная функция:
      - Превращает выходную структуру редьюсеров [(partition_id, [((i,k), val), ...]), ...]
        в матрицу numpy размера (I x K).

    :param reduce_output: список данных с партиций.
    :return: матрица (I x K).
    """
    mat = np.empty(shape=(I, K))
    for (_, partition_data) in reduce_output:
        for ((i, k), val) in partition_data:
            mat[i, k] = val
    return mat

result_matrix = asmatrix(partitioned_output)

print("Матрица M:\n", M)
print("\nМатрица N:\n", N)
print("\nРезультат распределённого MapReduce (M x N):\n", result_matrix)
print("\nЭталонное умножение NumPy:\n", reference_solution)
print("\nСовпадают ли результаты:", np.allclose(reference_solution, result_matrix))


480 key-value pairs were sent over a network.
Матрица M:
 [[0.18614802 0.43086229 0.50541163]
 [0.02727125 0.48784624 0.09553536]]

Матрица N:
 [[0.55167727 0.92142804 0.2834218  0.55662993 0.40344942 0.15833182
  0.38296548 0.29322887 0.43341875 0.83228835 0.52286038 0.52967013
  0.07862256 0.93416472 0.88052644 0.2006839  0.24002765 0.53620268
  0.9657826  0.66064828 0.61062801 0.13708736 0.7188866  0.92746106
  0.30552445 0.15955219 0.78075    0.25316753 0.55693648 0.91515754
  0.55211123 0.15609293 0.86766918 0.46533504 0.19980854 0.24198183
  0.6107798  0.05901447 0.44096832 0.23305812]
 [0.51066001 0.06973845 0.02206269 0.65644433 0.80110145 0.61280415
  0.98203424 0.40490953 0.24720452 0.19692432 0.65024078 0.75525434
  0.71876298 0.74683071 0.92792574 0.7217015  0.75783453 0.72504417
  0.09774496 0.29056237 0.11213156 0.22171899 0.43225411 0.60784539
  0.80298373 0.49460381 0.1686847  0.06163624 0.18141381 0.76651292
  0.96087996 0.29734645 0.87159851 0.13279574 0.03726932 0.71